# Pre-processamento de dados: exemplo e exercicio

Este notebook tem duas partes. Primeiro, execute um exemplo completo de preparacao de dados de clientes. Depois, aplique a mesma ideia em uma variacao com dados de leads.

**Objetivo:** preparar as entradas de um modelo sem deixar informacoes do conjunto de teste influenciarem o treino.

In [10]:
import numpy as np  # Importa NumPy para trabalhar com numeros e np.nan.
import pandas as pd  # Importa Pandas para criar e manipular tabelas.
from sklearn.compose import ColumnTransformer  # Combina tratamentos de tipos de coluna diferentes.
from sklearn.impute import SimpleImputer  # Preenche valores ausentes.
from sklearn.model_selection import train_test_split  # Separa os dados em treino e teste.
from sklearn.pipeline import Pipeline  # Encadeia etapas de transformacao.
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # Codifica texto e padroniza numeros.

## Parte 1: exemplo funcional

O conjunto possui valores ausentes em `idade` e `salario`, uma coluna de texto (`cidade`) e o alvo `comprou`.

In [11]:
def criar_dados_clientes():  # Define uma funcao que monta a tabela de exemplo.
    return pd.DataFrame(  # Cria um DataFrame a partir de um dicionario.
        {
            'idade': [22, 35, np.nan, 46, 28, 52, 31, np.nan, 40, 27, 38, 44],
            'salario': [3200, 5400, 4100, 6800, np.nan, 7200, 4900, 3900, 6100, 150000, 5800, 6500],
            'cidade': ['SP', 'RJ', 'SP', 'BH', 'RJ', 'SP', 'BH', 'RJ', 'SP', 'BH', 'RJ', 'SP'],
            'comprou': [0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1],
        }
    )

clientes = criar_dados_clientes()  # Executa a funcao e guarda a tabela.
clientes  # Exibe a tabela criada.

,idade,salario,cidade,comprou
0,22.0,3200.0,SP,0
1,35.0,5400.0,RJ,1
2,NaN,4100.0,SP,0
3,46.0,6800.0,BH,1
4,28.0,NaN,RJ,0
5,52.0,7200.0,SP,1
6,31.0,4900.0,BH,0
7,NaN,3900.0,RJ,0
8,40.0,6100.0,SP,1
9,27.0,150000.0,BH,1


In [12]:
print('Valores ausentes por coluna:')  # Mostra um titulo para a saida.
display(clientes.isna().sum())  # isna marca vazios; sum conta os vazios por coluna.

print('Resumo das colunas numericas:')  # Mostra um titulo para a saida.
display(clientes[['idade', 'salario']].describe())  # describe calcula estatisticas descritivas.

Valores ausentes por coluna:


idade      2
salario    1
cidade     0
comprou    0
dtype: int64

Resumo das colunas numericas:


,idade,salario
count,10.000000,11.000000
mean,36.300000,18536.363636
std,9.463967,43620.001667
min,22.000000,3200.000000
25%,28.750000,4500.000000
50%,36.500000,5800.000000
75%,43.000000,6650.000000
max,52.000000,150000.000000


### Separar antes de transformar

`X` contem as entradas e `y` contem o alvo. A divisao acontece antes de preencher, escalar ou codificar. Assim, o conjunto de teste permanece desconhecido durante o ajuste do pre-processador.

In [13]:
X = clientes.drop(columns='comprou')  # drop remove o alvo das entradas.
y = clientes['comprou']  # Seleciona a coluna alvo que queremos prever.

X_train, X_test, y_train, y_test = train_test_split(  # Divide entradas e alvo em treino e teste.
    X, y, test_size=0.25, random_state=42, stratify=y  # Reserva 25%, fixa o sorteio e preserva as classes.
)  # Finaliza a divisao.

print('Treino:', X_train.shape, 'Teste:', X_test.shape)  # shape mostra linhas e colunas de cada parte.

Treino: (9, 3) Teste: (3, 3)


### Criar o pre-processador

Para numeros, usamos a mediana para preencher ausentes e `StandardScaler` para colocar idade e salario em escalas comparaveis. Para cidade, usamos a categoria mais frequente quando necessario e one-hot encoding para criar colunas de 0 e 1.

In [14]:
numerico = Pipeline(  # Cria o fluxo de tratamento das colunas numericas.
    steps=[  # Lista as etapas executadas na ordem.
        ('imputar', SimpleImputer(strategy='median')),  # Preenche vazios com a mediana.
        ('escalar', StandardScaler()),  # Padroniza os valores para uma escala comparavel.
    ]  # Fecha a lista de etapas.
)  # Finaliza o pipeline numerico.

categorico = Pipeline(  # Cria o fluxo de tratamento das colunas de texto.
    steps=[  # Lista as etapas executadas na ordem.
        ('imputar', SimpleImputer(strategy='most_frequent')),  # Preenche vazios com a categoria mais frequente.
        ('codificar', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),  # Cria colunas 0/1 e ignora categorias novas.
    ]  # Fecha a lista de etapas.
)  # Finaliza o pipeline categorico.

pre_processador = ColumnTransformer(  # Aplica cada pipeline somente nas colunas indicadas.
    transformers=[  # Lista os grupos de colunas e seus tratamentos.
        ('numericas', numerico, ['idade', 'salario']),  # Trata idade e salario como numeros.
        ('categoricas', categorico, ['cidade']),  # Trata cidade como categoria.
    ],  # Fecha a lista de transformadores.
    verbose_feature_names_out=False,  # Mantem nomes de colunas mais curtos na saida.
)  # Finaliza o pre-processador.

In [15]:
X_train_pronto = pre_processador.fit_transform(X_train)  # fit_transform aprende no treino e o transforma.
X_test_pronto = pre_processador.transform(X_test)  # transform aplica ao teste as regras ja aprendidas.

colunas_prontas = pre_processador.get_feature_names_out()  # Recupera os nomes gerados apos a transformacao.
treino_pronto = pd.DataFrame(X_train_pronto, columns=colunas_prontas, index=X_train.index)  # Reconstrui o treino como tabela.
teste_pronto = pd.DataFrame(X_test_pronto, columns=colunas_prontas, index=X_test.index)  # Reconstrui o teste como tabela.

print('Colunas apos o pre-processamento:', list(colunas_prontas))  # list deixa os nomes mais faceis de ler.
print('Formato do treino:', treino_pronto.shape)  # Confere o tamanho do treino pronto.
print('Formato do teste:', teste_pronto.shape)  # Confere o tamanho do teste pronto.
treino_pronto.head()  # head mostra as cinco primeiras linhas.

Colunas apos o pre-processamento: ['idade', 'salario', 'cidade_BH', 'cidade_RJ', 'cidade_SP']
Formato do treino: (9, 5)
Formato do teste: (3, 5)


,idade,salario,cidade_BH,cidade_RJ,cidade_SP
2,0.066602,-0.975648,0.0,0.0,1.0
1,-0.293047,0.043556,0.0,1.0,0.0
8,0.306367,0.592358,0.0,0.0,1.0
7,0.066602,-1.132449,0.0,1.0,0.0
3,1.025664,1.141160,1.0,0.0,0.0


### O que observar no exemplo

- `comprou` nao aparece nas colunas prontas, pois e o alvo.
- `cidade` virou uma coluna para cada cidade.
- treino e teste terminaram com o mesmo conjunto de colunas.
- o salario de 150000 foi mantido. Identificar um outlier nao significa remove-lo automaticamente.

## Parte 2: exercicio - variacao do exemplo

Agora o problema muda: queremos preparar dados de leads para prever a coluna `converteu`. Use a mesma sequencia do exemplo, mas adapte as colunas e justifique suas escolhas.

**Desafio:** antes de executar a proxima celula, identifique quais colunas sao numericas, quais sao categoricas e qual e o alvo.

In [16]:
def criar_dados_leads():  # Define uma funcao que monta a tabela da variacao.
    return pd.DataFrame(  # Cria um DataFrame a partir de um dicionario.
        {
            'visitas_site': [2, 8, 4, np.nan, 15, 3, 10, 6, 20, 1, 7, 12],
            'tempo_minutos': [3.5, 18.0, 7.0, 4.5, np.nan, 2.0, 22.0, 11.0, 35.0, 1.0, 13.0, 25.0],
            'origem': ['busca', 'indicacao', 'busca', 'anuncio', 'anuncio', 'busca', 'indicacao', 'busca', 'anuncio', 'busca', 'indicacao', 'anuncio'],
            'converteu': [0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1],
        }
    )

leads = criar_dados_leads()  # Executa a funcao e guarda a tabela.
leads  # Exibe a tabela criada.

,visitas_site,tempo_minutos,origem,converteu
0,2.0,3.5,busca,0
1,8.0,18.0,indicacao,1
2,4.0,7.0,busca,0
3,NaN,4.5,anuncio,0
4,15.0,NaN,anuncio,1
5,3.0,2.0,busca,0
6,10.0,22.0,indicacao,1
7,6.0,11.0,busca,0
8,20.0,35.0,anuncio,1
9,1.0,1.0,busca,0


### Tarefa

1. Execute `leads.isna().sum()` e descreva os problemas encontrados.
2. Crie `X_leads` sem a coluna `converteu` e `y_leads` com essa coluna.
3. Divida os dados em treino e teste com `test_size=0.25`, `random_state=42` e `stratify=y_leads`.
4. Adapte o pre-processador para as colunas `visitas_site`, `tempo_minutos` e `origem`.
5. Mostre a tabela pronta e confirme que treino e teste possuem as mesmas colunas.

A proxima celula e uma resolucao funcional. Tente escrever a sua antes de executa-la.

In [17]:
X = leads.drop(columns='converteu')  # drop remove o alvo das entradas.
y = leads['converteu']  # Seleciona a coluna alvo que queremos prever.

X_train, X_test, y_train, y_test = train_test_split(  # Divide entradas e alvo em treino e teste.
    X, y, test_size=0.25, random_state=42, stratify=y  # Reserva 25%, fixa o sorteio e preserva as classes.
)  # Finaliza a divisao.
print('Treino:', X_train.shape, 'Teste:', X_test.shape)  # shape mostra linhas e colunas de cada parte.



numerico = Pipeline(  # Cria o fluxo de tratamento das colunas numericas.
    steps=[  # Lista as etapas executadas na ordem.
        ('imputar', SimpleImputer(strategy='median')),  # Preenche vazios com a mediana.
        ('escalar', StandardScaler()),  # Padroniza os valores para uma escala comparavel.
    ]  # Fecha a lista de etapas.
)  # Finaliza o pipeline numerico.
categorico = Pipeline(  # Cria o fluxo de tratamento das colunas de texto.
    steps=[  # Lista as etapas executadas na ordem.
        ('imputar', SimpleImputer(strategy='most_frequent')),  # Preenche vazios com a categoria mais frequente.
        ('codificar', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),  # Cria colunas 0/1 e ignora categorias novas.
    ]  # Fecha a lista de etapas.
)  # Finaliza o pipeline categorico.
pre_processador = ColumnTransformer(  # Aplica cada pipeline somente nas colunas indicadas.
    transformers=[  # Lista os grupos de colunas e seus tratamentos.
        ('numericas', numerico, ['visitas_site', 'tempo_minutos']),  # Trata idade e salario como numeros.
        ('categoricas', categorico, ['origem']),  # Trata cidade como categoria.
    ],  # Fecha a lista de transformadores.
    verbose_feature_names_out=False,  # Mantem nomes de colunas mais curtos na saida.
)  # Finaliza o pre-processador.



X_train_pronto = pre_processador.fit_transform(X_train)  # fit_transform aprende no treino e o transforma.
X_test_pronto = pre_processador.transform(X_test)  # transform aplica ao teste as regras ja aprendidas.

colunas_prontas = pre_processador.get_feature_names_out()  # Recupera os nomes gerados apos a transformacao.
treino_pronto = pd.DataFrame(X_train_pronto, columns=colunas_prontas, index=X_train.index)  # Reconstrui o treino como tabela.
teste_pronto = pd.DataFrame(X_test_pronto, columns=colunas_prontas, index=X_test.index)  # Reconstrui o teste como tabela.

print('Colunas apos o pre-processamento:', list(colunas_prontas))  # list deixa os nomes mais faceis de ler.
print('Formato do treino:', treino_pronto.shape)  # Confere o tamanho do treino pronto.
print('Formato do teste:', teste_pronto.shape)  # Confere o tamanho do teste pronto.
treino_pronto.head()  # head mostra as cinco primeiras linhas.

Treino: (9, 3) Teste: (3, 3)
Colunas apos o pre-processamento: ['visitas_site', 'tempo_minutos', 'origem_anuncio', 'origem_busca', 'origem_indicacao']
Formato do treino: (9, 5)
Formato do teste: (3, 5)


,visitas_site,tempo_minutos,origem_anuncio,origem_busca,origem_indicacao
0,-1.200490,-0.992149,0.0,1.0,0.0
2,-0.857493,-0.669956,0.0,1.0,0.0
11,0.514496,0.987035,1.0,0.0,0.0
1,-0.171499,0.342649,0.0,0.0,1.0
3,0.000000,-0.900094,1.0,0.0,0.0


In [18]:
# Resolucao da variacao. Cubra esta celula durante a tentativa.
X_leads = leads.drop(columns='converteu')  # Remove o alvo para formar as entradas.
y_leads = leads['converteu']  # Seleciona a coluna alvo que queremos prever.

X_train_leads, X_test_leads, y_train_leads, y_test_leads = train_test_split(  # Divide as entradas e o alvo.
    X_leads, y_leads, test_size=0.25, random_state=42, stratify=y_leads  # Reserva 25%, fixa o sorteio e preserva as classes.
)  # Finaliza a divisao.

pre_processador_leads = ColumnTransformer(  # Combina os tratamentos numerico e categorico.
    transformers=[  # Lista os grupos de colunas e seus tratamentos.
        (  # Inicia a configuracao das colunas numericas.
            'numericas',  # Define o nome do grupo.
            Pipeline(steps=[  # Encadeia o preenchimento e a escala.
                ('imputar', SimpleImputer(strategy='median')),  # Preenche vazios com a mediana.
                ('escalar', StandardScaler()),  # Padroniza os valores numericos.
            ]),  # Finaliza o pipeline numerico.
            ['visitas_site', 'tempo_minutos'],  # Seleciona as colunas numericas.
        ),  # Finaliza a configuracao numerica.
        (  # Inicia a configuracao das colunas categoricas.
            'categoricas',  # Define o nome do grupo.
            Pipeline(steps=[  # Encadeia o preenchimento e a codificacao.
                ('imputar', SimpleImputer(strategy='most_frequent')),  # Preenche vazios com a categoria mais frequente.
                ('codificar', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),  # Cria colunas 0/1 e ignora categorias novas.
            ]),  # Finaliza o pipeline categorico.
            ['origem'],  # Seleciona a coluna categorica.
        ),  # Finaliza a configuracao categorica.
    ],  # Fecha a lista de transformadores.
    verbose_feature_names_out=False,  # Mantem nomes de colunas mais curtos.
)  # Finaliza o pre-processador.

treino_leads = pd.DataFrame(  # Cria uma tabela com o treino transformado.
    pre_processador_leads.fit_transform(X_train_leads),  # Aprende no treino e o transforma.
    columns=pre_processador_leads.get_feature_names_out(),  # Define os nomes das colunas geradas.
    index=X_train_leads.index,  # Mantem os indices originais do treino.
)  # Finaliza a tabela de treino.
teste_leads = pd.DataFrame(  # Cria uma tabela com o teste transformado.
    pre_processador_leads.transform(X_test_leads),  # Aplica ao teste as regras aprendidas no treino.
    columns=pre_processador_leads.get_feature_names_out(),  # Reutiliza os mesmos nomes de colunas.
    index=X_test_leads.index,  # Mantem os indices originais do teste.
)  # Finaliza a tabela de teste.

assert list(treino_leads.columns) == list(teste_leads.columns)  # Interrompe se as colunas forem diferentes.
print('Validacao concluida: treino e teste possuem as mesmas colunas.')  # Confirma que a verificacao passou.
treino_leads.head()  # Mostra as cinco primeiras linhas do treino pronto.

Validacao concluida: treino e teste possuem as mesmas colunas.


,visitas_site,tempo_minutos,origem_anuncio,origem_busca,origem_indicacao
0,-1.200490,-0.992149,0.0,1.0,0.0
2,-0.857493,-0.669956,0.0,1.0,0.0
11,0.514496,0.987035,1.0,0.0,0.0
1,-0.171499,0.342649,0.0,0.0,1.0
3,0.000000,-0.900094,1.0,0.0,0.0


## Reflexao final

- O que mudaria se uma nova origem aparecesse apenas no conjunto de teste?
    - O teste não poderia ser realizado por sair do treinamento do modelo. Mas caso a coluna seja adicionada no conjunto de treino, uma nova coluna seria criada devido à abordagem de One-hot Encoding.
- Por que o `handle_unknown='ignore'` é util nesse caso?
    - Porque não seria relevante criar uma coluna para vazios ou nulos e não seria preciso gerar um erro caso eles estivessem presentes na coluna.
- Se `tempo_minutos` tivesse um valor extremo, qual informacao de negocio voce buscaria antes de removê-lo?
    - Qual a origem do dado, qual o limite real para aquele dado, qual o objetivo(focar na rotina ou incluir eventos raros/extremos) da pesquisa e se já houveram ocorrências semelhantes registradas.